In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import polars as pl
from preprocess_dataset_for_training import create_training_data
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
import matplotlib.pyplot as plt
import pandas as pd
import copy
import re

In [8]:
# -------------------------------
# Reproducibility setup
# -------------------------------
seed = 42  
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
# torch.cuda.manual_seed_all(seed)  # if using multi-GPU

# For deterministic behavior (slower but exact reproducibility)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [9]:
# -------------------------------
# Device configuration
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [10]:
# -------------------------------
# Load and preprocess data
# -------------------------------
file = "joined_df_with_weather"
joined_df = pl.read_parquet(f"./data/{file}.parquet")
X, y = create_training_data(joined_df)

In [11]:
X_np = X.to_numpy()
y_np = y.to_numpy()  # shape: (samples, 15 targets)
del joined_df

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, shuffle=False)
del X_np, y_np

# # Feature scaling
# scaler_X = StandardScaler()
# X_train = scaler_X.fit_transform(X_train)
# X_test = scaler_X.transform(X_test)

# Target scaling (important for stable float32 regression)
scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test = scaler_y.transform(y_test)

# Convert to PyTorch tensors on CPU (we'll move batches to GPU)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)
del X_train, X_test, y_train, y_test

# -------------------------------
# DataLoaders for batching
# -------------------------------
batch_size = 256  
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

test_dataset = TensorDataset(X_test_t, y_test_t)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

feature_names = X.columns 
lag_groups = {}
for i, fname in enumerate(feature_names):
    match = re.match(r"(.+)_t.*", fname)  # your regex
    if match:
        base = match.group(1)  # e.g., "feature1"
        if base not in lag_groups:
            lag_groups[base] = []
        lag_groups[base].append(i)

# -------------------------------
# Define multi-output NN with per-group stats
# -------------------------------
class MultiOutputNN(nn.Module):
    def __init__(self, input_dim, output_dim, lag_groups):
        super().__init__()
        self.lag_groups = list(lag_groups.values())

        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
            """
            x: (B, F)    ← features (already globally standardized)
            Performs INSTANCE NORMALIZATION per lag group:
                normalized = (group - mean) / std
            But DOES NOT invert normalization on output, because y has its own scaler.
            """

            x_norm = x.clone()

            # -----------------------------
            # Per-group instance norm on X
            # -----------------------------
            for indices in self.lag_groups:
                group = x[:, indices]                         # (B, group_size)

                mean = group.mean(dim=1, keepdim=True)        # (B, 1)
                std  = group.std(dim=1, keepdim=True) + 1e-6  # (B, 1)

                x_norm[:, indices] = (group - mean) / std

            # -----------------------------
            # Feed normalized X into model
            # -----------------------------
            return self.model(x_norm)

input_dim = X_train_t.shape[1]
output_dim = y_train_t.shape[1]

def smape(y_true, y_pred, eps=1e-8):
    denom = (np.abs(y_true) + np.abs(y_pred)) + eps
    return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom, axis=0)

num_seeds = 1
num_targets = output_dim
batch_size = 256

# containers
mae_all = np.zeros((num_seeds, num_targets))
rmse_all = np.zeros((num_seeds, num_targets))
r2_all = np.zeros((num_seeds, num_targets))
smape_all = np.zeros((num_seeds, num_targets))

# -------------------------------
# Training loop (unchanged)
# -------------------------------
for s in range(num_seeds):
    seed = s
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    model_s = MultiOutputNN(input_dim, output_dim, lag_groups).to(device)
    optimizer_s = optim.Adam(model_s.parameters(), lr=1e-4)
    criterion_s = nn.HuberLoss(delta=1.0, reduction='mean')

    # reproducible shuffling for DataLoader
    gen = torch.Generator()
    gen.manual_seed(seed)
    train_loader_s = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=gen)
    test_loader_s = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # early stopping
    best_val_loss = float('inf')
    counter = 0
    patience = 5
    best_state = copy.deepcopy(model_s.state_dict())

    epochs = 100
    for epoch in range(1, epochs + 1):
        model_s.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader_s:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer_s.zero_grad()
            outputs = model_s(X_batch)
            loss = criterion_s(outputs, y_batch)
            loss.backward()
            optimizer_s.step()

            running_loss += loss.item() * X_batch.size(0)
        train_loss = running_loss / len(train_loader_s.dataset)

        # validation
        model_s.eval()
        val_loss_total = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader_s:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                outputs = model_s(X_batch)
                val_loss_total += criterion_s(outputs, y_batch).item() * X_batch.size(0)
        val_loss = val_loss_total / len(test_loader_s.dataset)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0
            best_state = copy.deepcopy(model_s.state_dict())
        else:
            counter += 1
            if counter >= patience:
                break

    # load best model and evaluate
    model_s.load_state_dict(best_state)
    model_s.eval()
    y_pred_list = []
    y_true_list = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader_s:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model_s(X_batch)
            y_pred_list.append(outputs.cpu())
            y_true_list.append(y_batch.cpu())

    y_pred_scaled = torch.cat(y_pred_list).numpy()
    y_true_scaled = torch.cat(y_true_list).numpy()

    # inverse transform
    y_pred = scaler_y.inverse_transform(y_pred_scaled)
    y_true = scaler_y.inverse_transform(y_true_scaled)

    # compute metrics per target
    for i in range(num_targets):
        mae_all[s, i] = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse_all[s, i] = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        r2_all[s, i] = r2_score(y_true[:, i], y_pred[:, i])
    smape_all[s, :] = smape(y_true, y_pred)


In [12]:
# aggregate: mean across seeds (per-target)
mae_mean_per_target = mae_all.mean(axis=0)
rmse_mean_per_target = rmse_all.mean(axis=0)
r2_mean_per_target = r2_all.mean(axis=0)
smape_mean_per_target = smape_all.mean(axis=0)

# also overall averages across targets
overall = {
    "MAE": mae_mean_per_target.mean(),
    "RMSE": rmse_mean_per_target.mean(),
    "R2": r2_mean_per_target.mean(),
    "SMAPE": smape_mean_per_target.mean()
}

# prepare DataFrame per target
target_names = list(y.columns) if hasattr(y, "columns") else [f"target_{i}" for i in range(num_targets)]
metrics_df = pd.DataFrame({
    "Target": target_names,
    "MAE": mae_mean_per_target,
    "RMSE": rmse_mean_per_target,
    "R2": r2_mean_per_target,
    "SMAPE": smape_mean_per_target
})

display(metrics_df)
print("Overall averages across targets (mean over seeds then targets):")
print(overall)


,Target,MAE,RMSE,R2,SMAPE
0,Negative Balancing Energy Unit Price for Balan...,5.615613e+12,5.615726e+12,-3.388542e+21,200.0
1,Positive Balancing Energy Unit Price for Balan...,2.013010e+12,2.013050e+12,-4.354220e+20,200.0
2,System Direction (kWh)_t+15min,1.141171e+15,1.141194e+15,-1.732982e+21,200.0
3,Negative Balancing Energy Unit Price for Balan...,1.522677e+12,1.522707e+12,-2.491444e+20,200.0
4,Positive Balancing Energy Unit Price for Balan...,2.813293e+12,2.813350e+12,-8.504830e+20,200.0
5,System Direction (kWh)_t+30min,2.759725e+15,2.759781e+15,-1.013425e+22,200.0
6,Negative Balancing Energy Unit Price for Balan...,1.311991e+12,1.312017e+12,-1.849662e+20,200.0
7,Positive Balancing Energy Unit Price for Balan...,4.523261e+12,4.523353e+12,-2.198539e+21,200.0
8,System Direction (kWh)_t+45min,1.524622e+15,1.524653e+15,-3.092884e+21,200.0
9,Negative Balancing Energy Unit Price for Balan...,6.819216e+12,6.819355e+12,-4.996835e+21,200.0


Overall averages across targets (mean over seeds then targets):
{'MAE': np.float64(434109411011242.7), 'RMSE': np.float64(434118212606651.8), 'R2': np.float64(-2.2548969872342993e+21), 'SMAPE': np.float64(200.0)}


In [13]:
# optional: save
metrics_df.to_csv(f"NN_{num_seeds}_seeds_Huber_instance_norm_{file}_11-27.csv", index=False)


In [26]:
# compute std across seeds (per-target)
mae_mean_per_target = mae_all.mean(axis=0)
mae_std_per_target = mae_all.std(axis=0)

rmse_mean_per_target = rmse_all.mean(axis=0)
rmse_std_per_target = rmse_all.std(axis=0)

r2_mean_per_target = r2_all.mean(axis=0)
r2_std_per_target = r2_all.std(axis=0)

smape_mean_per_target = smape_all.mean(axis=0)
smape_std_per_target = smape_all.std(axis=0)

# add std columns to existing metrics_df (keeps previous mean columns)
metrics_df["MAE_STD"] = mae_std_per_target
metrics_df["RMSE_STD"] = rmse_std_per_target
metrics_df["R2_STD"] = r2_std_per_target
metrics_df["SMAPE_STD"] = smape_std_per_target

# display table and print concise mean ± std per target
display(metrics_df)

for i, t in enumerate(target_names):
    print(
        f"{t}: MAE {mae_mean_per_target[i]:.3f} ± {mae_std_per_target[i]:.3f}, "
        f"RMSE {rmse_mean_per_target[i]:.3f} ± {rmse_std_per_target[i]:.3f}, "
        f"R2 {r2_mean_per_target[i]:.3f} ± {r2_std_per_target[i]:.3f}, "
        f"SMAPE {smape_mean_per_target[i]:.3f} ± {smape_std_per_target[i]:.3f}"
    )

,Target,MAE,RMSE,R2,SMAPE,MAE_STD,RMSE_STD,R2_STD,SMAPE_STD
0,Negative Balancing Energy Unit Price for Balan...,52.963331,97.446015,-0.020305,146.244790,0.288005,0.121159,0.002537,0.196548
1,Positive Balancing Energy Unit Price for Balan...,54.798435,96.845305,-0.007763,145.105295,0.335224,0.082052,0.001708,0.193288
2,System Direction (kWh)_t+15min,16378.946602,28104.808452,-0.051087,149.418857,13.476643,64.719275,0.004841,2.271957
3,Negative Balancing Energy Unit Price for Balan...,52.960868,97.444057,-0.020304,146.244640,0.281008,0.117620,0.002463,0.191349
4,Positive Balancing Energy Unit Price for Balan...,54.790819,96.844216,-0.007780,145.107825,0.316258,0.077425,0.001612,0.182111
5,System Direction (kWh)_t+30min,16379.816064,28106.086522,-0.051103,149.430808,13.588765,62.831446,0.004701,2.225740
6,Negative Balancing Energy Unit Price for Balan...,52.954541,97.448059,-0.020376,146.250029,0.271647,0.115395,0.002417,0.186354
7,Positive Balancing Energy Unit Price for Balan...,54.793136,96.844293,-0.007770,145.106909,0.297989,0.073138,0.001522,0.171795
8,System Direction (kWh)_t+45min,16378.199336,28097.367376,-0.050400,149.090064,10.569202,55.693122,0.004164,1.918860
9,Negative Balancing Energy Unit Price for Balan...,52.959316,97.449273,-0.020391,146.249000,0.321615,0.138390,0.002900,0.221865


Negative Balancing Energy Unit Price for Balance Groups (HUF/kWh)_t+15min: MAE 52.963 ± 0.288, RMSE 97.446 ± 0.121, R2 -0.020 ± 0.003, SMAPE 146.245 ± 0.197
Positive Balancing Energy Unit Price for Balance Groups (HUF/kWh)_t+15min: MAE 54.798 ± 0.335, RMSE 96.845 ± 0.082, R2 -0.008 ± 0.002, SMAPE 145.105 ± 0.193
System Direction (kWh)_t+15min: MAE 16378.947 ± 13.477, RMSE 28104.808 ± 64.719, R2 -0.051 ± 0.005, SMAPE 149.419 ± 2.272
Negative Balancing Energy Unit Price for Balance Groups (HUF/kWh)_t+30min: MAE 52.961 ± 0.281, RMSE 97.444 ± 0.118, R2 -0.020 ± 0.002, SMAPE 146.245 ± 0.191
Positive Balancing Energy Unit Price for Balance Groups (HUF/kWh)_t+30min: MAE 54.791 ± 0.316, RMSE 96.844 ± 0.077, R2 -0.008 ± 0.002, SMAPE 145.108 ± 0.182
System Direction (kWh)_t+30min: MAE 16379.816 ± 13.589, RMSE 28106.087 ± 62.831, R2 -0.051 ± 0.005, SMAPE 149.431 ± 2.226
Negative Balancing Energy Unit Price for Balance Groups (HUF/kWh)_t+45min: MAE 52.955 ± 0.272, RMSE 97.448 ± 0.115, R2 -0.020 ± 